In [10]:
%reset -f out
import gc, torch
gc.collect(); torch.mps.empty_cache()
print(f"MPS allocated: {torch.mps.current_allocated_memory()/1e9:.2f} GB")

Flushing output cache (0 entries)
MPS allocated: 8.06 GB


In [11]:
import gc, time, platform, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen3-4B"
device = "mps" # cuda - nvdia gpus, mps - apple silicon, cpu - always works

# Idempotent cleanup: makes re-running this cell safe.
for _n in ("model", "tok"):
    if _n in globals():
        del globals()[_n]
gc.collect()
torch.mps.empty_cache()

print(f"torch {torch.__version__} | device={device} | {platform.machine()}")

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to(device)
model.eval()

c = model.config
print(f"LAYERS={c.num_hidden_layers}  d_model={c.hidden_size}  vocab={c.vocab_size}")

msgs = [{"role": "user", "content": "In one sentence, what is 17 * 3?"}]
text = tok.apply_chat_template(
    msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
inputs = tok(text, return_tensors="pt").to(device)



torch 2.13.0 | device=mps | arm64


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 6733.90it/s]


RuntimeError: MPS backend out of memory (MPS allocated: 18.10 GiB, other allocations: 720.00 KiB, max allowed: 18.13 GiB). Tried to allocate 47.50 MiB on shared pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [12]:
t0 = time.time()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
dt = time.time() - t0

n_new = out.shape[1] - inputs.input_ids.shape[1]
print(tok.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True))
print(f"\n>>> {n_new} tokens in {dt:.1f}s = {n_new/dt:.1f} tok/s")

17 multiplied by 3 is 51.

>>> 12 tokens in 5.8s = 2.1 tok/s


In [14]:
import time, torch

# Warm-up — compile kernels, throw the timing away.
with torch.no_grad():
    _ = model.generate(**inputs, max_new_tokens=8, do_sample=False)

# Steady state. min_new_tokens forces it to keep going past EOS
# so we actually measure 200 tokens of work, not 12.
t0 = time.time()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, min_new_tokens=200,
                         do_sample=False)
dt = time.time() - t0
n = out.shape[1] - inputs.input_ids.shape[1]
print(f"{n} tokens in {dt:.1f}s = {n/dt:.1f} tok/s")

200 tokens in 26.0s = 7.7 tok/s


In [5]:
import time, torch
from transformers import TextStreamer

PROBLEM = ("Natalia sold clips to 48 of her friends in April, and then she sold "
           "half as many clips in May. How many clips did Natalia sell altogether "
           "in April and May?")
DIRECT_SUFFIX = ("\n\nRespond with only the final numeric answer and nothing else. "
                 "Do not show any reasoning.")

def run(label, question, thinking, max_new=512):
    msgs = [{"role": "user", "content": question}]
    text = tok.apply_chat_template(msgs, tokenize=False,
                                   add_generation_prompt=True,
                                   enable_thinking=thinking)
    ins = tok(text, return_tensors="pt").to(device)
    print("="*70)
    print(f"{label} | enable_thinking={thinking} | cap={max_new}")
    print("--- prompt tail ---");  print(repr(text[-120:]))
    print("--- streaming ---")
    streamer = TextStreamer(tok, skip_prompt=True, skip_special_tokens=False)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**ins, max_new_tokens=max_new,
                             do_sample=False, streamer=streamer)
    dt = time.time() - t0
    n = out.shape[1] - ins.input_ids.shape[1]
    print(f"\n>>> {n} tok | {dt:.0f}s | {n/dt:.1f} tok/s | hit_cap={n >= max_new}")

In [6]:
run("C: thinking off + direct instruction", PROBLEM + DIRECT_SUFFIX, thinking=False, max_new=128)

C: thinking off + direct instruction | enable_thinking=False | cap=128
--- prompt tail ---
'he final numeric answer and nothing else. Do not show any reasoning.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
--- streaming ---
144<|im_end|>

>>> 4 tok | 2s | 1.9 tok/s | hit_cap=False


In [7]:
run("B: thinking off", PROBLEM, thinking=False, max_new=512)

B: thinking off | enable_thinking=False | cap=512
--- prompt tail ---
'in May. How many clips did Natalia sell altogether in April and May?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
--- streaming ---
Natalia sold clips to **48 friends** in **April**.

In **May**, she sold **half as many** clips as in April:

$$
\frac{48}{2} = 24
$$

Now, add the number of clips sold in both months:

$$
48 + 24 = 72
$$

**Answer:** Natalia sold **72 clips** altogether in April and May.<|im_end|>

>>> 94 tok | 14s | 6.9 tok/s | hit_cap=False


In [9]:
run("A: full CoT", PROBLEM, thinking=True, max_new=1024)

A: full CoT | enable_thinking=True | cap=1024
--- prompt tail ---
'half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>\n<|im_start|>assistant\n'
--- streaming ---
<think>
Okay, let's see. Natalia sold clips to 48 friends in April. Then in May, she sold half as many as in April. The question is asking how many clips she sold altogether in April and May. Hmm, so I need to find the total number of clips sold in both months.

First, let me break it down. In April, she sold 48 clips. That part is straightforward. Then in May, she sold half as many as in April. So, half of 48. Let me calculate that. Half of 48 is 24, right? Because 48 divided by 2 is 24. So, in May, she sold 24 clips.

Now, to find the total number of clips sold in both months, I need to add the number of clips from April and May together. So that would be 48 (April) plus 24 (May). Let me do the addition. 48 plus 24... 40 plus 20 is 60, and 8 plus 4 is 12. So 60 plus 12 is 72. Th

In [12]:
import re, datasets

ds = datasets.load_dataset("openai/gsm8k", "main")
print(ds)
print("datasets", datasets.__version__)   # pin this in your README

def gold_gsm8k(ans_field):
    """GSM8K puts the final answer after '####'."""
    return norm(ans_field.split("####")[-1])

NUM = r"-?\d[\d,]*\.?\d*"

def norm(s):
    if s is None: return None
    s = s.replace(",", "").replace("$", "").strip().rstrip(".")
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(f)
    except ValueError:
        return None

def strip_think(text):
    """Only ever parse what comes AFTER the reasoning block."""
    if "</think>" in text:
        text = text.rsplit("</think>", 1)[1]
    return text.replace("<|im_end|>", "").strip()

def extract_answer(text):
    body = strip_think(text)
    # 1. \boxed{...} — most reliable
    for cand in reversed(re.findall(r"\\boxed\{([^{}]*)\}", body)):
        n = re.findall(NUM, cand)
        if n: return norm(n[-1])
    # 2. an explicit answer cue
    m = re.findall(rf"(?:answer|total|result)\D{{0,20}}({NUM})", body, re.I)
    if m: return norm(m[-1])
    # 3. fallback: last number in the response
    n = re.findall(NUM, body)
    return norm(n[-1]) if n else None

Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 630172.80 examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})
datasets 5.0.1


In [15]:
TESTS = [
    ("Thus the total is $\\boxed{72}$.<|im_end|>",                    "72"),
    ("**Answer:** Natalia sold **72 clips** altogether.<|im_end|>",   "72"),
    ("144<|im_end|>",                                                "144"),
    ("The answer is 42.",                                             "42"),
    ("42 apples",                                                     "42"),
    ("\\boxed{1,234}",                                             "1234"),
    ("<think>maybe 48? no, 24.</think>\n\nThe answer is 72.",         "72"),
    ("The answer is 72 clips, more than the 48 she sold in April.",   "72"),
    ("She lost $-5$ dollars. \\boxed{-5}",                            "-5"),
    ("\\boxed{3.50}",                                                "3.5"),
    ("I cannot determine this.",                                      None),

    ("\\boxed{\\frac{3}{2}}",                                        "1.5"),
    ("\\boxed{72 \\text{ clips}}",                                    "72"),
    ("First \\boxed{48}, no wait — \\boxed{72}",                      "72"),
    ("The answer is 25%",                                             "25"),
    ("Total cost: $18.00",                                            "18"),
    ("<think>48 + 24 = 72. Hmm, let me verify: 48 * 3 = 144",         None),  # truncated
]

for raw, want in TESTS:
    got = extract_answer(raw)
    print(f"{'ok ' if got == want else 'FAIL'} want={want!r:8} got={got!r:8} <- {raw[:45]!r}")

ok  want='72'     got='72'     <- 'Thus the total is $\\boxed{72}$.<|im_end|>'
ok  want='72'     got='72'     <- '**Answer:** Natalia sold **72 clips** altoget'
ok  want='144'    got='144'    <- '144<|im_end|>'
ok  want='42'     got='42'     <- 'The answer is 42.'
ok  want='42'     got='42'     <- '42 apples'
ok  want='1234'   got='1234'   <- '\\boxed{1,234}'
ok  want='72'     got='72'     <- '<think>maybe 48? no, 24.</think>\n\nThe answer '
ok  want='72'     got='72'     <- 'The answer is 72 clips, more than the 48 she '
ok  want='-5'     got='-5'     <- 'She lost $-5$ dollars. \\boxed{-5}'
ok  want='3.5'    got='3.5'    <- '\\boxed{3.50}'
ok  want=None     got=None     <- 'I cannot determine this.'
FAIL want='1.5'    got='2'      <- '\\boxed{\\frac{3}{2}}'
ok  want='72'     got='72'     <- '\\boxed{72 \\text{ clips}}'
ok  want='72'     got='72'     <- 'First \\boxed{48}, no wait — \\boxed{72}'
ok  want='25'     got='25'     <- 'The answer is 25%'
ok  want='18'     got='18'     <- 'To

In [17]:
from importlib.metadata import version
for p in ["torch", "transformers", "datasets", "math-verify",
          "latex2sympy2_extended", "antlr4-python3-runtime", "sympy"]:
    print(f"{p:24} {version(p)}")

torch                    2.13.0
transformers             5.14.1
datasets                 5.0.1
math-verify              0.9.0
latex2sympy2_extended    1.11.0
antlr4-python3-runtime   4.13.2
sympy                    1.14.0


In [19]:
from scoring import score_file, provenance, validate_gold
from analysis import load_scores, report

DIRECT_SUFFIX = ("\n\nRespond with only the final numeric answer and nothing else. "
                 "Do not show any reasoning.")

# condition name -> (enable_thinking, prompt_suffix)
CONDS = {
    "direct_intact":  (False, DIRECT_SUFFIX),
    "direct_ablated": (False, DIRECT_SUFFIX),
    "cot_intact":     (True,  ""),
    "cot_ablated":    (True,  ""),
    # random-direction controls
    "direct_random":  (False, DIRECT_SUFFIX),
    "cot_random":     (True,  ""),
}